# 10.10 正確描述怎麼比其他描述更接近？

# 第 10 章：圖片如何進入文字模型

前置：embedding、序列與Linear。圖片入口先用16×16合成RGB，不需外部權重；圖片是否被理解必須等語言／視覺基礎能力訓練後測試。程式在tiny_perceptron/multimodal.py。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：像把名字卡與照片配對**

對角線是正配對；重複描述可能造成假負例。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/contrastive.svg")))

**先想一想：**兩張一樣紅圓圖，各自文字都「紅圓」，應互斥當負例嗎？

把正確配對當分類答案，其餘batch文字當候選，使用CE教圖找文字。若多張圖共享同一句描述，它們可能成為假負例。

**把直覺寫成數學：**L_image=CE(similarity/τ, pairing_index)。τ與取樣temperature用途不同。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
scores = torch.tensor([[2.0, 0.0], [0.0, 2.0]], requires_grad=True)
targets = torch.arange(2)
loss = F.cross_entropy(scores / 0.5, targets)
loss.backward()
print(loss.item(), scores.grad)

**如何讀結果：**正配對分數收到提升方向，其他欄收到降低方向。實際重複描述需多正例處理或設計不同資料。

**只改一件事：**只把targets交換，觀察學習方向反轉。
